In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pystan 
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pickle
from tqdm.notebook import tqdm as tqdm
import math
from scipy.stats import gaussian_kde

In [ ]:
model = """
data {
    int N;                        // number of data
    int L;                        // number of strain
    int K;                        // number of repetition
    vector[N] t_heat;             // heating time
    vector[N] n_dilu;             // Dilution counts
    vector[N] V;                  // Aliquot volume
    int Nt_plate_count[N];              // Colony counts
    int<lower=1, upper=L> ID_strain[N];
    int<lower=1, upper=K> ID_rep[N];
}

parameters {
    real<lower=6, upper=10> LogC0[L, K];
    real<lower=0, upper=5> delta[L];
    real<lower=0, upper=2.5> power[L];
    
    real<lower=-5, upper=1.6> logdelta0;
    real<lower=-1, upper=1> logpower0;
    
    cholesky_factor_corr[2] corr_chol_rep;
    vector<lower=0>[2] sigma_rep_vec;
    
}

transformed parameters{
    vector[N] Nt_plate_pred;
    real C0[L, K];
	cholesky_factor_cov[2] cov_chol_rep;
    vector [2] Paras[L];
    vector [2] Paras0;
    
    cov_chol_rep = diag_pre_multiply(sigma_rep_vec,　corr_chol_rep);
    Paras0[1] = logdelta0;
    Paras0[2] = logpower0;
    
    for (l in 1:L){
        Paras[l,1]　=　log(delta[l]);
        Paras[l,2]　=　log(power[l]);
        for (k in 1:K){
            C0[l, k] = 10^LogC0[l, k];
        }
    }
    for(n in 1:N){
        Nt_plate_pred[n] = C0[ID_strain[n],ID_rep[n]]*V[n]*10^(-n_dilu[n]-6*(t_heat[n]/6/delta[ID_strain[n]])^power[ID_strain[n]]);
    }
}

model {
    for (n in 1:N){
        Nt_plate_count[n] ~ poisson(Nt_plate_pred[n]);
    }
    Paras ~ multi_normal_cholesky(Paras0,　cov_chol_rep);
    
    sigma_rep_vec ~ cauchy(0, 1);
    corr_chol_rep ~ lkj_corr_cholesky(1);
}

generated quantities {
    corr_matrix[2] corr;
    cov_matrix[2] cov;
    corr = multiply_lower_tri_self_transpose(corr_chol_rep);
    cov = multiply_lower_tri_self_transpose(cov_chol_rep);
}
"""
sm = pystan.StanModel(model_code=model)

with open('modified Weibull MVN.pkl', 'wb') as f:

    pickle.dump(sm, f)

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

...

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

In file included from /var/folders/f9/2qbwnnt53m1dp3z2wqzblq540000gn/T/tmpzs__jwyt/stanfit4anon_model_ab95d42331cdddd2b56792e2f5cccef6_64979679519602718.cpp:846:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan_fit.hpp:22:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/services/diagnose/diagnose.hpp:10:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/test_gradients.hpp:7:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/src/stan/model/log_prob_grad.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/mat.hpp:4:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/stan_math/stan/math/rev/core.hpp:5:
In file included from /opt/anaconda3/envs/pystan/lib/python3.7/site-packages/pystan/stan/lib/sta

167 warnings generated.
ld: warning: -pie being ignored. It is only used when linking a main executable


In [ ]:
data = pd.read_csv('Cam_all_55ºC_2024_01_31.csv')
data = data[(data["Strain_Name"]!="RIMD 0366027")&(data["Strain_Name"]!="RIMD 0366043")&(data["Strain_Name"]!="IFTK 344")]
data["ID_strain_II"] = (data["ID_strain"] != data["ID_strain"].shift(1)).cumsum()
data["ID_strain_origin"] = data["ID_strain"]
data["ID_strain"] = data["ID_strain_II"]
data = data.drop("ID_strain_II", axis=1)

pd.set_option('display.max_rows', None)
data

In [ ]:
StrainCount = 51
StrainCount_MPC = 51
datum = data[data["ID_strain"]<=StrainCount]
datum

In [ ]:
StrainNum_MPC = list(range(1,StrainCount+1))
print(StrainNum_MPC)

for i in range(StrainCount_MPC):
    if i == 0:
        datum_MPC =  data[data["ID_strain"]==StrainNum_MPC[i]]
    if i != 0:
        datum_MPC = pd.concat([datum_MPC, data[data["ID_strain"]==StrainNum_MPC[i]]], axis=0, ignore_index=True)

In [ ]:
datum_MPC["ID_strain_MPC"] = (datum_MPC["ID_strain"] != datum_MPC["ID_strain"].shift(1)).cumsum()
datum_MPC

In [ ]:
sm = pickle.load(open('modified Weibull MVN.pkl', 'rb'))

fit_nuts = sm.sampling(
    data = dict(N = len(datum_MPC['Nt_count']),
                L = int(StrainCount),
                K = 6,
                ID_strain = datum_MPC['ID_strain'],
                ID_rep = datum_MPC['rep_ID'],
                t_heat = datum_MPC['t_heat'],
                n_dilu = datum_MPC['n_dilu'],
                V = datum_MPC['V'],
                Nt_plate_count = datum_MPC['Nt_count']), 
    iter = 50000, chains = 4, thin = 1, warmup = 25000, seed = 12345,  control = dict(adapt_delta = 0.8, max_treedepth = 15))

In [ ]:
print(fit_nuts)

In [ ]:
samples = fit_nuts.extract(permuted=False, inc_warmup=True)
paraname = fit_nuts.sim["fnames_oi"]
palette = sns.color_palette()
ms = fit_nuts.extract(permuted=False, inc_warmup=True)
iter_from = fit_nuts.sim['warmup']
iter_range = np.arange(iter_from, ms.shape[0])
paraname = fit_nuts.sim['fnames_oi']
num_pages = math.ceil(len(paraname)/4)
for pg in tqdm(range(num_pages),desc='Progress', leave=False):
    plt.figure()
    for pos in range(4):
        pi = pg*4 + pos
        if pi >= len(paraname): break
        plt.subplot(4, 2, 2*pos+1)
        plt.tight_layout()
        [plt.plot(iter_range + 1, ms[iter_range,ci,pi], color=palette[ci]) for ci in range(ms.shape[1])]
        plt.title(paraname[pi])
        plt.subplot(4, 2, 2*(pos+1))
        plt.tight_layout()
        [sns.kdeplot(ms[iter_range,ci,pi], color=palette[ci]) for ci in range(ms.shape[1])]
        plt.title(paraname[pi])
    plt.show()

In [ ]:
samples = fit_nuts.extract(permuted=True)

In [ ]:
with open('Samples extracted modfied Weibull MVN stan.pkl', 'wb') as g:
    pickle.dump(samples, g)

In [ ]:
sm = pickle.load(open('modified Weibull MVN.pkl', 'rb'))
samples = pickle.load(open('Samples extracted modfied Weibull MVN stan.pkl', 'rb'))

In [ ]:
logParas_sim = np.empty([len(samples['delta'][:,0]), 2])
for k in range(len(samples['delta'][:,0])):
    logParas_sim[k,:] = np.random.multivariate_normal(samples["Paras0"][k,:], samples["cov"][k,:])

In [ ]:
Paras_sim = np.exp(logParas_sim)

In [ ]:
import scipy 
L = samples['Paras'][0,:,0].shape[0]
dev_logdelta = np.array([])
dev_logpower = np.array([])
for l in range(L):
    logdeltas_tmp = samples['Paras'][:,l, 0]
    logpowers_tmp = samples['Paras'][:,l, 1]
    dev_logdelta_tmp = logdeltas_tmp - np.mean(logdeltas_tmp)
    dev_logpower_tmp = logpowers_tmp - np.mean(logpowers_tmp)
    dev_logdelta = np.concatenate([dev_logdelta, dev_logdelta_tmp])
    dev_logpower = np.concatenate([dev_logpower, dev_logpower_tmp])

sigma_logdelta = np.std(dev_logdelta)
sigma_logpower = np.std(dev_logpower)
cor_paras = scipy.stats.pearsonr(dev_logdelta, dev_logpower)[0]

In [ ]:
print([sigma_logdelta, sigma_logpower, cor_paras])

In [ ]:
samples['cov'][0]

In [ ]:
Colmax = StrainCount + 1
Palette = sns.color_palette("hls", n_colors = Colmax)
plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("δ parameter",size=15,fontname="Arial")
plt.ylabel("p parameter",size=15,fontname="Arial")
#plt.ylim([0.995,1.255])
#plt.xlim([2.15,3.05])
plt.ylim([-0.5,5.5])
plt.xlim([-0.5,10.5])

plt.plot(Paras_sim[:,0],Paras_sim[:,1], color="gray", lw=0, marker='.', alpha=0.025)

for i in range(StrainCount):
    StNum = int(np.mean(datum["ID_strain"][datum["ID_strain"]==(i+1)]))-1
    plt.plot(samples['delta'][:,StNum],samples['power'][:,StNum], color=Palette[i], lw=0, marker='.', alpha=0.025)
plt.show()

In [ ]:
g = sns.jointplot(Paras_sim[:,0],Paras_sim[:,1], kind="kde", color="gray", levels = [0.01, 0.10, 0.5, 0.90, 0.99])
for i in range(StrainCount):
    StNum = int(np.mean(datum["ID_strain"][datum["ID_strain"]==(i+1)]))-1
    g.fig.axes[0].scatter(samples['delta'][:,StNum],samples['power'][:,StNum], color=Palette[i])
        
g.fig.axes[0].set(xlabel ='delta', ylabel='power', xlim=(-0.5,5.5), ylim=(-0.05,2.55))
g.fig.set_figheight(4.8*1.2)
g.fig.set_figwidth(6.4*1.2)
plt.show()

In [ ]:
g = sns.jointplot(logParas_sim[:,0],logParas_sim[:,1], color="gray", kind="kde", levels = [0.01, 0.10, 0.5, 0.90, 0.99])
for i in range(StrainCount):
    StNum = int(np.min(datum["ID_strain"][datum["ID_strain"]==(i+1)]))-1
    g.fig.axes[0].scatter(np.log(samples['delta'][:,StNum]),np.log(samples['power'][:,StNum]), color=Palette[i])
g.fig.axes[0].set(xlabel ='ln delta', ylabel='ln power', xlim=(-1,2), ylim=(-1,1))
g.fig.set_figheight(4.8*1.2)
g.fig.set_figwidth(6.4*1.2)
plt.show()

In [ ]:
wholedelta = samples['delta'][:,0]
wholepower = samples['power'][:,0]
for i in range(StrainCount_fit):
    wholedelta = np.concatenate([wholedelta, samples['delta'][:,i]])
    wholepower = np.concatenate([wholepower, samples['power'][:,i]])

In [ ]:
plt.figure(figsize=(14,7))
plt.hist(wholedelta, alpha=0.3, bins=300, density=True)
plt.hist(Paras_sim[:,0], alpha=0.3, bins=450, density=True)
sns.kdeplot(wholedelta)
sns.kdeplot(Paras_sim[:,0])
plt.xlim(0,4)
plt.show()

In [ ]:
StrainName = list(data["Strain_Name"].unique())
plt.figure(figsize=(14,7))
for i in range(StrainCount):
    StNum = int(np.mean(datum_fit["ID_strain"][datum_fit["ID_strain"]==(i+1)]))-1
    plt.hist(samples['delta'][:,StNum], alpha=0.3, bins=150, density=True, color=Palette[i])
    sns.kdeplot(samples['delta'][:,StNum], color=Palette[i], label=StrainName[i])
plt.xlim(0,4)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14,7))
plt.hist(wholedelta, alpha=0.3, bins=100, density=True, cumulative=True)
plt.hist(Paras_sim[:,0], alpha=0.3, bins=750, density=True, cumulative=True)
sns.kdeplot(wholedelta, cumulative=True)
sns.kdeplot(Paras_sim[:,0], cumulative=True)
plt.xlim(0,4)
plt.show()

In [ ]:
plt.figure(figsize=(14,7))
dens_wholedelta, X_delta = np.histogram(wholedelta, range = (0,4), bins = 400, density = True)
dx_delta = X_delta[1]-X_delta[0]
cumu_wholedelta = np.append(0, np.cumsum(dens_wholedelta)*dx_delta)
dens_preddelta, X_delta2 = np.histogram(Paras_sim[:,0], range = (0,40), bins = 4000, density = True)
dx_delta = X_delta2[1]-X_delta2[0]
cumu_preddelta = np.append(0, np.cumsum(dens_preddelta)*dx_delta)
cumu_preddelta = cumu_preddelta[:len(cumu_wholedelta)]
plt.plot(X_delta, ((cumu_wholedelta-cumu_preddelta)**2)**0.5)
np.mean(((cumu_wholedelta-cumu_preddelta)**2)**0.5)

In [ ]:
print(np.quantile(wholedelta,[0.025,0.975]), np.quantile(Paras_sim[:,0],[0.025,0.975]))

In [ ]:
plt.figure(figsize=(14,7))
plt.hist(wholepower, alpha=0.3, bins=200, density=True)
plt.hist(Paras_sim[:,1], alpha=0.3, bins=300, density=True)
sns.kdeplot(wholepower)
sns.kdeplot(Paras_sim[:,1])
plt.xlim(0,2.5)
plt.show()

In [ ]:
plt.figure(figsize=(14,7))
plt.hist(wholepower, alpha=0.3, bins=100, density=True, cumulative=True)
plt.hist(Paras_sim[:,1], alpha=0.3, bins=750, density=True, cumulative=True)
sns.kdeplot(wholepower, cumulative=True)
sns.kdeplot(Paras_sim[:,1], cumulative=True)
plt.xlim(0,2.5)
plt.show()

In [ ]:
plt.figure(figsize=(14,7))
for i in range(StrainCount):
    if i+1 not in StrainNum_Vali:
        StNum = int(np.mean(datum_fit["ID_strain"][datum_fit["ID_strain"]==(i+1)]))-1
        plt.hist(samples['power'][:,StNum], alpha=0.3, bins=150, density=True, color=Palette[i])
        sns.kdeplot(samples['power'][:,StNum], color=Palette[i], label=StrainName[i])
plt.xlim(0,2.5)
plt.legend()
plt.show()

In [ ]:
dens_wholepower, X_power = np.histogram(wholepower, range = (0,2), bins = 200, density = True)
dx_power = X_power[1]-X_power[0]
cumu_wholepower = np.append(0, np.cumsum(dens_wholepower)*dx_power)
dens_predpower, X_power2 = np.histogram(Paras_sim[:,1], range = (0,20), bins = 2000, density = True)
dx_power = X_power2[1]-X_power2[0]
cumu_predpower = np.append(0, np.cumsum(dens_predpower)*dx_power)
cumu_predpower = cumu_predpower[:len(cumu_wholepower)]
plt.plot(X_power, ((cumu_wholepower-cumu_predpower)**2)**0.5)
np.mean(((cumu_wholepower-cumu_predpower)**2)**0.5)

In [ ]:
len(cumu_predpower[:len(cumu_wholepower)])

In [ ]:
print(np.quantile(wholepower,[0.025,0.975]), np.quantile(Paras_sim[:,1],[0.025,0.975]))

In [ ]:
datum_fit = data
data

In [ ]:
time = [0.0, 2.25, 4.5, 6.75, 9.0, 11.25]
Means = np.empty([StrainCount, 6])
Stds = np.empty([StrainCount, 6])
for l in range(StrainCount):
    mean = np.empty(6)
    std = np.empty(6)
    for i in range(6):
        mean[i] = np.mean(datum[(datum['ID_strain'] == l+1)&(datum['t_heat'] == time[i])]['LogSt'].astype('float'))
        std[i] = np.std(datum[(datum['ID_strain'] == l+1)&(datum['t_heat'] == time[i])]['LogSt'].astype('float'))
        Means[l,:] = mean
        Stds[l,:] = std

In [ ]:
def Fit_Model(t, delta, power) :
    return -6*(t/6/delta)**power

In [ ]:

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")
plt.ylim([-10.5,0.5])

time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in range(len(time_interval)):
    for k in range(Num_mcmc):
        logS_credible_mcmc[j,k] = Fit_Model(time_interval[j], 
                                            Paras_sim[k,0], Paras_sim[k,1])
    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])


plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor="gray", alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color="gray", linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color="gray", linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color="gray")

for i in tqdm(range(1, StrainCount+1),desc='Progress', leave=False):
    StrainNum = i
    
    
    time_min = 0
    time_max = 12
    devide = (time_max-time_min)*10
    time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                              (time_max-time_min)/devide, dtype="float")
    Num_mcmc = len(samples["lp__"])
    logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
    logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)
    
    for j in range(len(time_interval)):
        for k in range(Num_mcmc):
                StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==(i)]))-1
                logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

        logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])
        
    plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.show()

In [ ]:
zerodata = datum[datum['non_detect']==1]
zerodata

In [ ]:
Colmax = StrainCount + 1
Palette = sns.color_palette("hls", n_colors = Colmax)

In [ ]:
for StrainNum in tqdm(range(1,51),desc='Progress', leave=False):
    time_min = 0
    time_max = 12
    devide = (time_max-time_min)*10
    time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                              (time_max-time_min)/devide, dtype="float")
    Num_mcmc = len(samples["lp__"])
    logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
    logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)
    
    for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
        for k in range(Num_mcmc):
                StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
                logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])
    
        logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])
    
    StrainName = datum_fit["Strain_Name"][datum_fit["ID_strain"]==StrainNum].head(1).iloc[0]
    plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
    plt.rcParams["font.family"] = "Arial"
    
    plt.ylim([-10.5,0.5])
    plt.xticks(size=14)
    plt.yticks(size=14)
    plt.xlabel("Time (min)",size=15,fontname="Arial")
    plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")
    
    
    plt.plot(
        time, Means[StrainNum-1,:], 
        linestyle='-', 
        marker='.', 
        ms=10, 
        color=Palette[StrainNum-1], 
        label=StrainName
    )
    plt.errorbar(
        time,Means[StrainNum-1,:],
        yerr=list(Stds[StrainNum-1,:]),
        fmt="none",
        color=Palette[StrainNum-1],
        capsize = 5
    )
    
    zerotime = zerodata[zerodata["ID_strain"]== StrainNum]["t_heat"]
    zerolimit = zerodata[zerodata["ID_strain"]== StrainNum]["LogSt_lim"]

    plt.plot(
        zerotime, zerolimit, 
        linestyle='', 
        marker='x', 
        ms=10,
        color=Palette[StrainNum-1]
    )
    
    plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="99% credible range")
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])
    
    plt.legend()
    if StrainName=="lA525":
        plt.savefig("Figures/single MPC IA525.pdf", bbox_inches="tight", dpi=150)
    else:
        plt.savefig("Figures/single MPC "+StrainName+".pdf", bbox_inches="tight", dpi=150)
    plt.show()


In [ ]:
for StrainNum in tqdm([6],desc='Progress', leave=False):
    time_min = 0
    time_max = 12
    devide = (time_max-time_min)*10
    time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                              (time_max-time_min)/devide, dtype="float")
    Num_mcmc = len(samples["lp__"])
    logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
    logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)
    
    for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
        for k in range(Num_mcmc):
                StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
                logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])
    
        logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])
    
    StrainName = datum_fit["Strain_Name"][datum_fit["ID_strain"]==StrainNum].head(1).iloc[0]
    plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
    plt.rcParams["font.family"] = "Arial"
    
    plt.xticks(size=14)
    plt.yticks(size=14)
    plt.xlabel("Time (min)",size=15,fontname="Arial")
    plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")
    
    
    plt.plot(
        time, Means[StrainNum-1,:], 
        linestyle='-', 
        marker='.', 
        ms=10, 
        color=Palette[StrainNum-1], 
        label=StrainName
    )
    plt.errorbar(
        time,Means[StrainNum-1,:],
        yerr=list(Stds[StrainNum-1,:]),
        fmt="none",
        color=Palette[StrainNum-1],
        capsize = 5
    )
    
    zerotime = zerodata[zerodata["ID_strain"]== StrainNum]["t_heat"]
    zerolimit = zerodata[zerodata["ID_strain"]== StrainNum]["LogSt_lim"]

    plt.plot(
        zerotime, zerolimit, 
        linestyle='', 
        marker='x', 
        ms=10,
        color=Palette[StrainNum-1]
    )
    
    
    plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="99% credible range")
    plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])
    
    plt.legend()
    plt.savefig("Figures/single MPC "+StrainName+".pdf", bbox_inches="tight", dpi=150)
    plt.show()


In [ ]:
StrainName

In [ ]:
StrainNum = 2


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 3


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 4

time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 5


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 6


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 7


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 8


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 9


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 10


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 11


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 12


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 13


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 14


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 15


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 16


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 17


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 18


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 19


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 20


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 21


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 22


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 23


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 24


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 25


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 26


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 27


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 28


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 29


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 30


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
StrainNum = 31


time_min = 0
time_max = 12
devide = (time_max-time_min)*10
time_interval = np.arange(time_min, time_max*(devide+1)/devide,
                          (time_max-time_min)/devide, dtype="float")
Num_mcmc = len(samples["lp__"])
logS_credible_mcmc = np.zeros((len(time_interval),Num_mcmc), dtype=float)
logS_credible_mcmc_quantile = np.zeros((len(time_interval), 3), dtype=float)

for j in tqdm(range(len(time_interval)),desc='Progress', leave=False):
    for k in range(Num_mcmc):
        if StrainNum not in StrainNum_Vali:
            StNum = int(np.min(datum_fit["ID_strain"][datum_fit["ID_strain"]==StrainNum]))-1
            logS_credible_mcmc[j,k] = Fit_Model(time_interval[j],samples["delta"][k, StNum], samples["power"][k, StNum])

    logS_credible_mcmc_quantile[j,:] = np.quantile(logS_credible_mcmc[j,:],[0.005,  0.5, 0.995])

plt.figure(figsize=[6.4*1.2, 4.8*1.2], dpi=150)
plt.rcParams["font.family"] = "Arial"

plt.xticks(size=14)
plt.yticks(size=14)
plt.xlabel("Time (min)",size=15,fontname="Arial")
plt.ylabel("Log reduction (log10 CFU/CFU)",size=15,fontname="Arial")

plt.fill_between(time_interval, logS_credible_mcmc_quantile[:,0], logS_credible_mcmc_quantile[:,2], facecolor=Palette[StrainNum-1], alpha=0.2)
plt.plot(time_interval, logS_credible_mcmc_quantile[:,0], lw=1, color=Palette[StrainNum-1], linestyle='--')
plt.plot(time_interval, logS_credible_mcmc_quantile[:,2], lw=1, color=Palette[StrainNum-1], linestyle='--', label="Log S credible")
plt.plot(time_interval, logS_credible_mcmc_quantile[:,1], lw=1, color=Palette[StrainNum-1])

plt.plot(
    time, Means[StrainNum-1,:], 
    linestyle='-', 
    marker='.', 
    ms=10, 
    color=Palette[StrainNum-1], 
    label=StrainName[StrainNum-1]
)
plt.errorbar(
    time,Means[StrainNum-1,:],
    yerr=list(Stds[StrainNum-1,:]),
    fmt="none",
    color=Palette[StrainNum-1],
    capsize = 5
)
plt.legend()
plt.show()


In [ ]:
print(sm)